# Hypothesis Testing (Two-Proportion Z-Test)

ใช้ Two-Proportion Z-Test เพื่อทดสอบว่า Accuracy ที่ลดลงจาก 67.41% เป็น 58.46% นั้น "แตกต่างกันอย่างมีนัยสำคัญทางสถิติ" หรือไม่ (ไม่ใช่แค่ลดลงเพราะความบังเอิญ)

### v4.5
```
========== FINAL EVALUATION ==========
Total Samples : 1979
Accuracy      : 0.6741
Precision     : 0.4815
Recall        : 0.4555
F1 Score      : 0.4626

========== CONFUSION MATRIX ==========
          Occlusal  Mesial  Distal  Other
Occlusal        98     149     104     37
Mesial          13     559      68     30
Distal         129      68     677     47
Other            0       0       0      0

========== CLASSIFICATION REPORT ==========
              precision    recall  f1-score   support

    Occlusal       0.41      0.25      0.31       388
      Mesial       0.72      0.83      0.77       670
      Distal       0.80      0.74      0.76       921
       Other       0.00      0.00      0.00         0

    accuracy                           0.67      1979
   macro avg       0.48      0.46      0.46      1979
weighted avg       0.70      0.67      0.68      1979
```

=============

### v4.6 — Diagonal-from-Centroid (4-triangle zone split)
```
========== FINAL EVALUATION ==========
Total Samples : 1979
Accuracy      : 0.5846
Precision     : 0.4938
Recall        : 0.4559
F1 Score      : 0.4466

========== CONFUSION MATRIX ==========
          Occlusal  Mesial  Distal  Other
Occlusal       243      72      36     37
Mesial         100     504      36     30
Distal         412      52     410     47
Other            0       0       0      0

========== CLASSIFICATION REPORT ==========
              precision    recall  f1-score   support

    Occlusal       0.32      0.63      0.43       388
      Mesial       0.80      0.75      0.78       670
      Distal       0.85      0.45      0.58       921
       Other       0.00      0.00      0.00         0

    accuracy                           0.58      1979
   macro avg       0.49      0.46      0.45      1979
weighted avg       0.73      0.58      0.62      1979
```

In [ ]:
import numpy as np
from statsmodels.stats.proportion import proportions_ztest

# 1. กำหนดข้อมูลจากผล Evaluation
n_total = 1979

# จำนวนเคสที่ทายถูก (คำนวณจากแนวทแยงของ Confusion Matrix)
# v4.5: Occlusal(98) + Mesial(559) + Distal(677)
correct_v45 = 98 + 559 + 677 

# v4.6: Occlusal(243) + Mesial(504) + Distal(410)
correct_v46 = 243 + 504 + 410

# 2. ตั้งสมมติฐาน (Hypothesis)
# H0: ความแม่นยำของ v4.5 และ v4.6 ไม่แตกต่างกัน
# H1: ความแม่นยำของ v4.5 และ v4.6 แตกต่างกัน

count = np.array([correct_v45, correct_v46])
nobs = np.array([n_total, n_total])

# 3. คำนวณค่าสถิติ Z-test
stat, pval = proportions_ztest(count, nobs)

print("========== ผลการทดสอบทางสถิติ ==========")
print(f"v4.5 Accuracy: {correct_v45/n_total:.4f} ({correct_v45}/{n_total})")
print(f"v4.6 Accuracy: {correct_v46/n_total:.4f} ({correct_v46}/{n_total})")
print(f"Z-Statistic: {stat:.4f}")
print(f"P-Value: {pval:.4e}")
print("========================================")

if pval < 0.05:
    print("สรุป: ปฏิเสธสมมติฐานหลัก (H0)")
    print("ความแม่นยำของโมเดล v4.5 สูงกว่า v4.6 'อย่างมีนัยสำคัญทางสถิติ' ที่ระดับความเชื่อมั่น 95%")
else:
    print("สรุป: ยอมรับสมมติฐานหลัก (H0) ความแม่นยำไม่แตกต่างกันอย่างมีนัยสำคัญ")
    
    
# ค่า P-Value จะออกมาน้อยกว่า 0.0001 มากๆ ซึ่งแปลว่า v4.5 ดีกว่าอย่างมีนัยสำคัญแน่นอน

========== ผลการทดสอบทางสถิติ ==========
v4.5 Accuracy: 0.6741 (1334/1979)
v4.6 Accuracy: 0.5846 (1157/1979)
Z-Statistic: 5.8252
P-Value: 5.7052e-09
สรุป: ปฏิเสธสมมติฐานหลัก (H0)
ความแม่นยำของโมเดล v4.5 สูงกว่า v4.6 'อย่างมีนัยสำคัญทางสถิติ' ที่ระดับความเชื่อมั่น 95%


# รายงานเปรียบเทียบและประเมินผลอัลกอริทึมจำแนกพื้นผิวฟันผุ (v4.5 vs v4.6)

ในการพัฒนาระบบระบุตำแหน่งรอยผุ (Dental Caries Surface Classification) ได้มีการทดลองเปรียบเทียบอัลกอริทึมการแบ่งโซน 2 รูปแบบ ได้แก่ **v4.5 (แบ่งแถบแนวตั้ง X-thirds)** และ **v4.6 (แบ่งเส้นทแยงมุม Diagonal-from-Centroid)** เพื่อหาโมเดลที่เหมาะสมที่สุด

---

## 1. สมมติฐานของการทดลอง (Hypothesis)
เราตั้งสมมติฐานว่า: *"การเปลี่ยนวิธีแบ่งโซนรอยผุจากแบบแถบตรงแนวตั้ง (v4.5) มาเป็นแบบแผ่รัศมีเส้นทแยงมุมจากจุดศูนย์กลาง (v4.6) จะช่วยแก้ปัญหารอยผุที่เกิดบริเวณมุมโค้งของหัวฟัน และทำให้ความแม่นยำ (Accuracy) รวมของโมเดลสูงขึ้น"*

## 2. ผลการทดสอบทางสถิติ (Statistical Results)
จากการประเมินผลบนชุดข้อมูลทดสอบทั้งหมด 1,979 ตัวอย่าง พบผลลัพธ์ดังนี้:
* **v4.5 (Vertical X-thirds):** Accuracy = 67.41%
* **v4.6 (Diagonal-from-Centroid):** Accuracy = 58.46%

จากการทดสอบสมมติฐานด้วย **Two-Proportion Z-Test** พบว่าค่า P-Value < 0.001 ซึ่งแปลผลได้ว่า **โมเดล v4.5 มีความแม่นยำสูงกว่าโมเดล v4.6 อย่างมีนัยสำคัญทางสถิติ** ที่ระดับความเชื่อมั่น 95% คณะผู้จัดทำจึงตัดสินใจเลือก v4.5 เป็นโมเดลหลักในเฟสนี้

---

## 3. การวิเคราะห์เชิงเรขาคณิตและสาเหตุของความคลาดเคลื่อน (Geometric & Data Analysis)
แม้สมมติฐานของ v4.6 จะดูสมเหตุสมผลในทางกายวิภาคฟัน แต่จาก Confusion Matrix พบสาเหตุเชิง Data Science ที่ทำให้ผลลัพธ์สวนทางกับสมมติฐาน ดังนี้:

### 3.1 ปรากฏการณ์ "Tall Bounding Box" (ทำไม v4.6 ถึงคะแนนตก?)
* **ปัญหา:** เนื่องจากภาพ Segmentation ของฟันเป็นการดึงกรอบภาพ (Bounding Box) ที่ **รวมรากฟัน (Root) เข้าไปด้วย** ทำให้กรอบภาพมีลักษณะเป็น "สี่เหลี่ยมผืนผ้าทรงสูง" (สูงกว่ากว้าง)
* **ผลกระทบ:** เมื่ออัลกอริทึม v4.6 ตีเส้นกากบาททะแยงมุมในกรอบทรงสูง พื้นที่สามเหลี่ยมด้านบน (Occlusal) จึงถูกยืดออกให้กว้างมากจนไป "กินพื้นที่" สามเหลี่ยมด้านข้าง (Mesial/Distal)
* **หลักฐาน:** ใน v4.6 รอยผุที่เป็นคลาส Distal จำนวนถึง **412 เคส** (เกือบ 50% ของ Distal ทั้งหมด) ถูกทำนายผิดว่าเป็น Occlusal อย่างรุนแรง

### 3.2 ปัญหา Data Imbalance (ทำไม v4.5 ถึงคะแนนสูง?)
* **ปัญหา:** ชุดข้อมูลมีสภาวะ Imbalance อย่างหนัก โดยมีคลาส Mesial (670) และ Distal (921) รวมกันถึง 1,591 เคส (คิดเป็น **80%** ของข้อมูลทั้งหมด) ในขณะที่มีคลาส Occlusal เพียง 20%
* **ผลกระทบ:** อัลกอริทึม v4.5 ถูกตั้งค่าให้มีพื้นที่ดักจับด้านข้างกว้างถึงฝั่งละ 40% ซึ่งสอดคล้องกับ "คลาสส่วนใหญ่ (Majority Class)" ของ Dataset พอดี ทำให้กวาดคะแนน Accuracy ไปได้มาก 
* **ข้อควรระวัง:** แม้ v4.5 จะคะแนนรวมสูง แต่แลกมากับความสามารถในการตรวจจับ Occlusal ที่ต่ำมาก (Recall เพียง 0.25)

### 3.3 ข้อดีที่ซ่อนอยู่ของ v4.6
แม้คะแนนรวมจะลดลง แต่อัลกอริทึม v4.6 สามารถเพิ่มประสิทธิภาพในการตรวจจับคลาส Occlusal ได้ดีขึ้นจริง (Recall เพิ่มขึ้นจาก 0.25 ใน v4.5 เป็น **0.63** ใน v4.6) ซึ่งพิสูจน์ได้ว่า Logic การแผ่รัศมีจากศูนย์กลางสามารถดักจับรอยผุมุมโค้งได้ตามที่ตั้งสมมติฐานไว้ เพียงแต่ถูกรบกวนด้วยสัดส่วนของรากฟัน

---

## 4. สรุปและแนวทางพัฒนาต่อ (Future Work)
ในสภาวะปัจจุบันที่ Bounding box ครอบคลุมไปถึงรากฟัน **อัลกอริทึม v4.5 ถือเป็นทางเลือกที่เสถียรที่สุด** จึงถูกเลือกนำมาใช้งาน

**แผนการปรับปรุงในอนาคต (Next Steps):**
หากต้องการดึงศักยภาพสูงสุดของอัลกอริทึม v4.6 (Diagonal-split) ออกมา จำเป็นต้องเพิ่มกระบวนการ **Crown-Root Segmentation** เพื่อตัดรากฟันออกก่อนทำการจัดหมวดหมู่ ซึ่งจะทำให้กรอบของตัวฟัน (Crown) มีสัดส่วนใกล้เคียงสี่เหลี่ยมจัตุรัสมากขึ้น ส่งผลให้การแบ่งโซน 4 สามเหลี่ยมสมดุล และแก้ปัญหาการ Over-predict ของคลาส Occlusal ได้อย่างสมบูรณ์